<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/NeuroLearn/blob/main/SlideMDIOWordClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [589]:
# Standard Imports
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [590]:
### HYPERPARAMETERS
MAX_WORD_LEN = 30
BN_EMA_SF = 0.95
LR = 0.001
N_EMBD = 10
NUM_OF_NEURONS_L1 = 128
BATCH_SIZE = 32
NUM_OF_NEURONS_L2 = 64
NUM_OF_NEURONS_L3 = 16
EPOCHS = 4

In [591]:
# "Opening" the Dataset
# To Be Replaced w/ real dataset & preprocessing
df = pd.read_csv("slidemdtestdata.csv")

In [592]:
# String to Integer Index
stoi = {chr(i): i - ord('a') + 1 for i in range(ord('a'), ord('z')+1)}
stoi['.'] = 0 # Padding Char
itos = {ix: char for char, ix in stoi.items()}

In [593]:
# Encoding the Words
def tok_word(word):
  # Array of chars
  letters = list(word)
  enc_letter = lambda letter: stoi[letter] if letter in stoi.keys() else 0
  enc_word = list(map(enc_letter, letters))

  enc_word_len = len(enc_word)
  if enc_word_len < MAX_WORD_LEN:
    enc_word.extend([0] * (MAX_WORD_LEN - enc_word_len))

  return enc_word

In [594]:
# Tokenizing Words in Dataset
inputs = df['word'].tolist()
X = np.array(list(map(tok_word, inputs)))
Y = np.array(df['is_medical']).reshape(-1, 1)

In [595]:
# Training / Test Split
n = int(X.shape[0] * 0.8)
Xtr, Xte = X[:n], X[n:]
Ytr, Yte = Y[:n], Y[n:]

In [596]:
### Classes for Neural Network Micrograd Style
class Embedding:
  def __init__ (self, size, n_embd):
    self.n_embd = n_embd
    self.C = np.random.randn(*size, n_embd)
    self.dC = np.zeros_like(self.C)

  def __call__ (self, X):
    self.X = X
    self.out = self.C[self.X]
    return self.out

  def _backwards(self, dup):
    np.add.at(self.dC, self.X.ravel(), dup.reshape(-1, self.n_embd))
    self.dout = dup
    return self.dout

  def _parameters(self):
    return [(self.C, self.dC)]

  def _wipegrad(self):
    self.dC = np.zeros_like(self.C)

class Reshape:
  def __init__ (self, shape):
    self.shape = shape

  def __call__ (self, X):
    self.X = X
    self.out = self.X.reshape(self.shape)
    return self.out

  def _backwards(self, dup):
    self.dout = dup.reshape(self.X.shape)
    return self.dout

  def _parameters(self):
    return []

  def _wipegrad(self):
    pass

class Linear:
  def __init__ (self, size):
    self.W = np.random.randn(*size) * 0.01
    self.B = np.zeros(size[-1])
    self.dW = np.zeros_like(self.W)
    self.dB = np.zeros_like(self.B)

  def __call__ (self, X):
    self.X = X
    self.out = self.X @ self.W + self.B
    return self.out

  def _backwards (self, dup):
    # print(f"deriv of up grad shape: {dup.shape}")
    # print(f"input shape: {self.X.shape}")
    # print(f"weight shape: {self.W.shape}") # I could figure this out, but j for ezier access
    # print(f"bias shape: {self.B.shape}") # same reason
    self.dW += self.X.T @ dup
    self.dB += dup.sum(axis = (tuple(range(self.B.ndim))))
    self.dout = dup @ self.W.T
    return self.dout # The issue driving me insane was I forgot this ONE LINE

  def _parameters(self):
    return [(self.W, self.dW), (self.B, self.dB)]

  def _wipegrad (self):
    self.dW = np.zeros_like(self.W)
    self.dB = np.zeros_like(self.B)

class BatchNorm:
  def __init__(self, size, training=True):
    self.training = training
    self.g = np.ones((size))
    self.b = np.zeros((size))
    self.dg = np.zeros_like(self.g)
    self.db = np.zeros_like(self.b)
    self.running_mean = None
    self.running_var = None

  def __call__ (self, X):
    if self.training:
      # Calculate New Stats
      self.xmean = np.mean(X, axis = 0, keepdims = True)
      self.xvar = np.var(X, axis = 0, keepdims = True)

      # Assign First Run to First Means/Vars
      if self.running_mean is None and self.running_var is None:
        self.running_mean = self.xmean
        self.running_var = self.xvar
      else:
        self.running_mean = BN_EMA_SF * self.running_mean + (1 - BN_EMA_SF) * self.xmean
        self.running_var = BN_EMA_SF * self.running_var + (1 - BN_EMA_SF) * self.xvar
    else:
      self.xmean = self.running_mean
      self.xvar = self.running_var

    self.X = X # Saving Input for BP
    # Batch Norm
    self.raw = (self.X - self.xmean)/np.sqrt(self.xvar + 1e-5)
    self.out = self.g * self.raw + self.b
    return self.out

  def _backwards(self, dup):
    # print(f"Input Type: {type(self.X)}, {self.X.shape}")
    # print(f"Deriv of Up Grad Type: {type(dup)}, {dup.shape}")
    # print(f"Gamma Shape: {self.g.shape}")
    # print(dup)
    self.dg = np.sum(dup * self.X, axis = tuple(range(0, self.g.ndim, 1)))
    # print(f"Input Gamma (b4 sum): {(dup * self.X).shape}")
    # print(tuple(range(0, self.g.ndim, 1)))
    self.db = np.sum(dup, axis = tuple(range(0, self.b.ndim, 1)))
    # print(f"Input Beta (b4 sum): {(dup).shape}")
    # print(tuple(range(0, self.b.ndim, 1)))
    self.dout = ((dup * self.dg) - (dup * self.dg).mean(axis=1, keepdims=True) - self.raw * ((dup * self.dg) * self.raw).mean(axis=1, keepdims=True))/(np.sqrt(self.xvar + 1e-5))
    return self.dout

  def _parameters(self):
    return [(self.g, self.dg), (self.b, self.db)]

  def _wipegrad(self):
    self.dg = np.zeros_like(self.g)
    self.db = np.zeros_like(self.b)

class LeakyReLu:
  def __call__ (self, X):
    self.X = X # saved for bp
    self.out = np.maximum(0.01 * self.X, self.X)
    return self.out

  def _backwards(self, dup):
    # print(dup)
    bg_deriv = np.ones_like(dup) * 0.01
    self.dout = np.where(self.X >= 0, dup, bg_deriv)
    return self.dout

  def _parameters(self):
    return []

  def _wipegrad(self):
    pass

class Sigmoid:
  def __call__ (self, X):
    self.X = X # svd for bp
    # print(self.X)
    self.out = (1 + np.e ** -(self.X) + 1e-5) ** -1
    return self.out

  def _backwards(self, dup):
    self.dout = (np.e ** self.X) * (1 + np.e ** -(self.X) + 1e-5) ** -2
    return self.dout

  def _parameters(self):
    return []

  def _wipegrad(self):
    pass

class Model:
  def __init__(self, layers, lr):
    self.layers = layers
    self.lr = lr

  def __call__(self, X):
    self.X = X
    self.out = self.X
    for layer in self.layers:
      self.out = layer(self.out)
      print(f"{layer.__class__.__name__} {self.layers.index(layer)}: {self.out.mean()}")
    return self.out

  def backwards(self, dup):
    self.dout = dup
    for layer in reversed(self.layers):
      # print(f"B4 {layer.__class__.__name__}: {self.dout}")
      self.dout = layer._backwards(self.dout)
      # print(f"After {layer.__class__.__name__}: {self.dout}")

  def update_parameters(self):
    parameters = []
    for layer in self.layers:
      parameters.extend(layer._parameters())

      # print(layer.__class__.__name__)
      # if len(layer._parameters()) > 0:
      #   for parameter, gradient in layer._parameters():
      #     print(parameter.shape, gradient.shape)

    for p,g in parameters:
      p -= g * self.lr

  def wipegrad(self):
    for layer in self.layers:
      layer.__wipegrad()


In [597]:
# My model
layers = [Embedding((X.shape[0],), N_EMBD), Reshape((BATCH_SIZE, -1)),
          Linear((MAX_WORD_LEN * N_EMBD, NUM_OF_NEURONS_L1)), BatchNorm((NUM_OF_NEURONS_L1)), LeakyReLu(),
          Linear((NUM_OF_NEURONS_L1, NUM_OF_NEURONS_L2)), BatchNorm((NUM_OF_NEURONS_L2)), LeakyReLu(),
          Linear((NUM_OF_NEURONS_L2, NUM_OF_NEURONS_L3)), BatchNorm((NUM_OF_NEURONS_L3)), LeakyReLu(),
          Linear((NUM_OF_NEURONS_L3, 1)), Sigmoid()]

nn = Model(layers, LR)

In [598]:
for epoch in range(EPOCHS):
  for i in range(0, Xtr.shape[0] // BATCH_SIZE):
    # Mini-Batching
    start = i * BATCH_SIZE
    end = i * BATCH_SIZE + BATCH_SIZE
    xbatch = Xtr[start: end]
    ybatch = Ytr[start: end]

    # Forward Pass
    rawprobs = nn(xbatch)
    probs = np.clip(rawprobs, 1e-7, 1 - 1e-7)
    loss = (-(ybatch * np.log(probs) + (1 - ybatch) * np.log(1 - probs))).mean()

    # Backward Pass
    dloss = ((probs - ybatch) / (probs * (1 - probs))) / BATCH_SIZE
    print("dloss mean", dloss.mean())
    nn.backwards(dloss)

    # Update Parameters
    nn.update_parameters()

  print(f"{epoch}: {loss}")

Embedding 0: -0.1660142579339614
Reshape 1: -0.1660142579339614
Linear 2: -0.003459290379612654
BatchNorm 3: -6.5052130349130266e-18
LeakyReLu 4: 0.3891930770208731
Linear 5: -0.0003849077353581229
BatchNorm 6: -1.734723475976807e-17
LeakyReLu 7: 0.3892308660731844
Linear 8: 0.006831845536742458
BatchNorm 9: -3.469446951953614e-18
LeakyReLu 10: 0.3868523456389944
Linear 11: 0.013450637604154173
Sigmoid 12: 0.5033592547435202
dloss mean -0.0034860398657915164
Embedding 0: -0.1473437211768753
Reshape 1: -0.1473437211768753
Linear 2: -0.002317403540668381
BatchNorm 3: -0.00015866132673587018
LeakyReLu 4: 0.3947685558277477
Linear 5: 5.7112850480897355e-05
BatchNorm 6: -0.0001606019206218971
LeakyReLu 7: 0.39144079122405256
Linear 8: 0.006425566013730727
BatchNorm 9: -0.00016073070616930314
LeakyReLu 10: 0.39596650090778124
Linear 11: -0.013111161738146714
Sigmoid 12: 0.49672013338937354
dloss mean -0.0004124093681363728
Embedding 0: -0.15065665540770992
Reshape 1: -0.15065665540770992
Lin

/tmp/ipykernel_4039/718185417.py:148: RuntimeWarning: overflow encountered in power
  self.out = (1 + np.e ** -(self.X) + 1e-5) ** -1
/tmp/ipykernel_4039/718185417.py:152: RuntimeWarning: overflow encountered in power
  self.dout = (np.e ** self.X) * (1 + np.e ** -(self.X) + 1e-5) ** -2
/tmp/ipykernel_4039/718185417.py:52: RuntimeWarning: invalid value encountered in add
  self.out = self.X @ self.W + self.B
/tmp/ipykernel_4039/718185417.py:62: RuntimeWarning: invalid value encountered in matmul
  self.dout = dup @ self.W.T


BatchNorm 3: nan
LeakyReLu 4: nan
Linear 5: nan
BatchNorm 6: nan
LeakyReLu 7: nan
Linear 8: nan
BatchNorm 9: nan
LeakyReLu 10: nan
Linear 11: nan
Sigmoid 12: nan
dloss mean nan
Embedding 0: nan
Reshape 1: nan
Linear 2: nan
BatchNorm 3: nan
LeakyReLu 4: nan
Linear 5: nan
BatchNorm 6: nan
LeakyReLu 7: nan
Linear 8: nan
BatchNorm 9: nan
LeakyReLu 10: nan
Linear 11: nan
Sigmoid 12: nan
dloss mean nan
Embedding 0: nan
Reshape 1: nan
Linear 2: nan
BatchNorm 3: nan
LeakyReLu 4: nan
Linear 5: nan
BatchNorm 6: nan
LeakyReLu 7: nan
Linear 8: nan
BatchNorm 9: nan
LeakyReLu 10: nan
Linear 11: nan
Sigmoid 12: nan
dloss mean nan
Embedding 0: nan
Reshape 1: nan
Linear 2: nan
BatchNorm 3: nan
LeakyReLu 4: nan
Linear 5: nan
BatchNorm 6: nan
LeakyReLu 7: nan
Linear 8: nan
BatchNorm 9: nan
LeakyReLu 10: nan
Linear 11: nan
Sigmoid 12: nan
dloss mean nan
Embedding 0: nan
Reshape 1: nan
Linear 2: nan
BatchNorm 3: nan
LeakyReLu 4: nan
Linear 5: nan
BatchNorm 6: nan
LeakyReLu 7: nan
Linear 8: nan
BatchNorm 9:

In [599]:
out = nn(X[0:BATCH_SIZE])
loss = (-(Y[0:BATCH_SIZE] * np.log(out) + (1 - Y[0:BATCH_SIZE]) * np.log(1 - out))).mean()
loss

Embedding 0: nan
Reshape 1: nan
Linear 2: nan
BatchNorm 3: nan
LeakyReLu 4: nan
Linear 5: nan
BatchNorm 6: nan
LeakyReLu 7: nan
Linear 8: nan
BatchNorm 9: nan
LeakyReLu 10: nan
Linear 11: nan
Sigmoid 12: nan


np.float64(nan)

In [600]:
dloss = 1/BATCH_SIZE * (-Y[0:BATCH_SIZE]/out + (1 - Y[0:BATCH_SIZE])/(1 - out))

In [601]:
nn.backwards(dloss)

In [602]:
nn.update_parameters()